# Regularisation and callbacks

A network with enough weights can memorise its training set. Training loss keeps falling while the loss on unseen data turns round and climbs. Regularisation is anything that stops that. This notebook covers L2 weight decay, dropout and early stopping, then the Keras callbacks that make training runs manageable: checkpointing, learning-rate-on-plateau and CSV logging.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import tempfile
from pathlib import Path

from common.dl import set_seed, setup_keras

setup_keras("torch")
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common.datasets import load_mnist
from common.plotting import plot_history

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

A student who memorises last year's exam answers scores full marks on last year's paper and fails this year's. A model that memorises its training set does the same. The signs are a training loss that keeps falling and a validation loss that stops falling and rises.

Three ways to stop memorisation: penalise large weights (L2), make units unreliable so no single path can carry a memorised answer (dropout), or stop training at the best moment (early stopping).

## 2. Maths and cost function

**L2** adds a penalty on the squared weights to the loss:

$$J_{reg}=J+\frac{\lambda}{2}\sum_i w_i^2\quad\Rightarrow\quad w\leftarrow w-\eta\,(\nabla J+\lambda w)$$

Each step shrinks every weight by a factor $(1-\eta\lambda)$ on top of the usual gradient. It is the same ridge penalty as in section 03. Keras's `l2(0.01)` adds $0.01\sum w^2$ with no half.

**Dropout** zeroes each unit with probability $p$ during training and scales the rest by $1/(1-p)$ ("inverted" dropout), so the expected activation is unchanged. At test time it does nothing.

**Early stopping** watches validation loss and stops after it has failed to improve for `patience` epochs.

## 3. From scratch

Inverted dropout is three lines. Check that the mean activation survives, and that individual activations are noisy.

In [ ]:
def dropout(h: np.ndarray, p: float, rng: np.random.Generator) -> np.ndarray:
    keep = rng.random(h.shape) >= p
    return h * keep / (1 - p)


h = np.abs(rng.normal(size=(10_000, 1)))
for p in (0.0, 0.3, 0.5, 0.8):
    out = dropout(h, p, rng)
    print(f"p={p}: mean {h.mean():.3f} -> {out.mean():.3f}, share zeroed {(out == 0).mean():.2f}")

The mean stays at 0.80 while the share of zeroed units tracks $p$. The scaling is what lets you switch dropout off at test time with no further adjustment. Early stopping is a loop over the validation losses:

In [ ]:
def early_stop_epoch(val_losses: list[float], patience: int) -> tuple[int, int]:
    """Return (best_epoch, stop_epoch), 0-indexed."""
    best, best_epoch = np.inf, 0
    for epoch, loss in enumerate(val_losses):
        if loss < best:
            best, best_epoch = loss, epoch
        elif epoch - best_epoch >= patience:
            return best_epoch, epoch
    return best_epoch, len(val_losses) - 1


print(early_stop_epoch([1.0, 0.8, 0.7, 0.72, 0.71, 0.75, 0.8], patience=3))

The best epoch is 2 and training stops at epoch 5, three epochs after the last improvement. We will compare this against Keras below.

## 4. Library version

To make overfitting easy to see, train on only 300 MNIST images with a wide net (two layers of 512 units). Four variants: none, L2, dropout, and both.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(300, 2000, seed=SEED)
x_tr, x_te = x_tr.reshape(len(x_tr), -1), x_te.reshape(len(x_te), -1)


def make_net(l2: float = 0.0, dropout_rate: float = 0.0) -> keras.Model:
    set_seed(SEED)
    reg = keras.regularizers.l2(l2) if l2 else None
    layers = [keras.Input((784,))]
    for _ in range(2):
        layers.append(keras.layers.Dense(512, activation="relu", kernel_regularizer=reg))
        if dropout_rate:
            layers.append(keras.layers.Dropout(dropout_rate))
    layers.append(keras.layers.Dense(10, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


variants = {
    "none": {},
    "L2 1e-3": {"l2": 1e-3},
    "dropout 0.5": {"dropout_rate": 0.5},
    "both": {"l2": 1e-3, "dropout_rate": 0.5},
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharey=True)
for ax, (name, kw) in zip(axes, variants.items()):
    hist = (
        make_net(**kw)
        .fit(x_tr, y_tr, epochs=20, batch_size=128, validation_data=(x_te, y_te), verbose=0)
        .history
    )
    plot_history(hist, ("loss", "val_loss"), ax, title=name)
    print(
        f"{name:>12}: train acc {hist['accuracy'][-1]:.3f}, val acc {hist['val_accuracy'][-1]:.3f}, min val loss {min(hist['val_loss']):.3f}, final val loss {hist['val_loss'][-1]:.3f}"
    )
plt.show()

With no regularisation the training accuracy is 100% and the validation loss climbs from 0.48 at its best to 0.58 at the end. Dropout flattens that climb (0.46 to 0.50). The L2 panels sit higher because Keras adds the penalty term to the loss it reports, so compare their shape, not their height. On accuracy all four land between 85.0% and 85.5%, inside the noise of a 2,000-image test set. Regularisation here is a small gain, not a rescue.

Now the callbacks. `EarlyStopping` with `restore_best_weights` rolls back to the best epoch. `ReduceLROnPlateau` cuts the learning rate when validation loss stalls. `ModelCheckpoint` saves the best weights as they appear. `CSVLogger` writes one row per epoch to a file, which is a light stand-in for TensorBoard (point `keras.callbacks.TensorBoard(log_dir)` at a folder and run `tensorboard --logdir` if you want the dashboard). Everything goes to a temporary folder, so nothing lands in the repo.

In [ ]:
workdir = Path(tempfile.mkdtemp())
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
    keras.callbacks.ModelCheckpoint(
        workdir / "best.weights.h5", save_best_only=True, save_weights_only=True
    ),
    keras.callbacks.CSVLogger(workdir / "log.csv"),
]
model = make_net()
hist = model.fit(
    x_tr,
    y_tr,
    epochs=60,
    batch_size=128,
    validation_data=(x_te, y_te),
    verbose=0,
    callbacks=callbacks,
).history
log = pd.read_csv(workdir / "log.csv")
print("epochs run:", len(log), "| best epoch (0-indexed):", int(log["val_loss"].idxmin()))
print("checkpoint written:", (workdir / "best.weights.h5").exists())
print(log[["epoch", "loss", "val_loss", "learning_rate"]].tail(4).round(4).to_string(index=False))
print("from-scratch rule:", early_stop_epoch(hist["val_loss"], patience=3))
print("final val acc (best weights restored):", round(model.evaluate(x_te, y_te, verbose=0)[1], 3))

The stopping rule written above gives the same answer as Keras: best epoch 6, stop at epoch 9, so 10 epochs ran instead of 60. The learning rate column shows `ReduceLROnPlateau` halving the rate from 0.001 as progress stalled.

## 5. Tuning

Dropout rate and L2 strength each need a sweep. Report validation accuracy at the last epoch and the gap to training accuracy.

In [ ]:
print("dropout rate")
for rate in (0.0, 0.2, 0.5, 0.8):
    h = (
        make_net(dropout_rate=rate)
        .fit(x_tr, y_tr, epochs=20, batch_size=128, validation_data=(x_te, y_te), verbose=0)
        .history
    )
    print(
        f"  {rate}: val acc {h['val_accuracy'][-1]:.3f}, train-val gap {h['accuracy'][-1] - h['val_accuracy'][-1]:.3f}"
    )
print("L2 strength")
for lam in (0.0, 1e-4, 1e-3, 1e-2):
    h = (
        make_net(l2=lam)
        .fit(x_tr, y_tr, epochs=20, batch_size=128, validation_data=(x_te, y_te), verbose=0)
        .history
    )
    print(
        f"  {lam}: val acc {h['val_accuracy'][-1]:.3f}, train-val gap {h['accuracy'][-1] - h['val_accuracy'][-1]:.3f}"
    )

Moderate dropout narrows the gap (0.149 to 0.112 at rate 0.5) with a gain of about half a point. L2 up to 0.001 changes almost nothing. At dropout 0.8 the gap goes negative because training accuracy is measured with units dropped, and validation accuracy falls to 78%. The next cell shows the far end for L2.

## 6. What breaks

Too much regularisation. L2 at 0.1 puts a penalty on the weights that rivals the data loss.

In [ ]:
h = (
    make_net(l2=0.1)
    .fit(x_tr, y_tr, epochs=20, batch_size=128, validation_data=(x_te, y_te), verbose=0)
    .history
)
print(
    f"L2 0.1: train acc {h['accuracy'][-1]:.3f}, val acc {h['val_accuracy'][-1]:.3f}, train loss {h['loss'][-1]:.2f}"
)

Validation accuracy drops from 85% to 77%, worse than no regularisation at all, and the reported loss of 6.0 is mostly penalty. Too weak a regulariser leaves a large train-validation gap. Too strong a one costs accuracy on both sides. Sweep the strength on a log scale and pick by validation accuracy, not by the gap.